In [ ]:
# improved_qml_pipeline.py
# Keep Qiskit usage compatible with qiskit ~1.4.4 (uses primitives and qiskit_machine_learning.SPSA)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix

# Qiskit imports (kept as in your original)
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.optimizers import SPSA

# -----------------------
# User configuration
# -----------------------
FILE_NAME = "Heart_data.csv"
TEST_SIZE = 0.2
RANDOM_SEED = 24
PCA_COMPONENTS = 3          # reduce to 3 qubits
FEATURE_MAP_REPS = 1        # smaller reps
VAR_ANSATZ_REPS = 1         # lightweight ansatz
SPSA_MAXITER = 600          # increase iterations
SCALE_TO_PI = True          # scale inputs to [0, pi]

# -----------------------
# 1) Load and normalize data
# -----------------------
df = pd.read_csv(FILE_NAME)
print(f"SUCCESS: Data loaded from {FILE_NAME}")

FEATURE_COLUMNS = ['Age', 'Gender', 'BloodPressure', 'Cholesterol', 'HeartRate', 'QuantumPatternFeature']
X = df[FEATURE_COLUMNS].values.astype(float)
y = df['HeartDisease'].astype(int).values

print("\n--- Data Head ---")
print(df.head())
print("\n--- Array Dimensions ---")
print(f"Features (X) shape: {X.shape}")
print(f"Labels (y) shape: {y.shape}")

# Min-Max scale to [0,1]
X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_normalized = (X - X_min) / (X_max - X_min)
X_normalized[np.isnan(X_normalized)] = 0.0

# Optionally scale to [0, pi] for feature map inputs
if SCALE_TO_PI:
    X_for_featuremap = X_normalized * np.pi
else:
    X_for_featuremap = X_normalized.copy()

# -----------------------
# 2) Dimensionality reduction (PCA) -> reduces qubits and stabilizes training
# -----------------------
pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_SEED)
X_reduced = pca.fit_transform(X_for_featuremap)

print(f"\nAfter PCA: X_reduced shape = {X_reduced.shape}")
N_FEATURES = X_reduced.shape[1]
print(f"Using N_FEATURES (qubits) = {N_FEATURES}")

# -----------------------
# 3) Train-test split (use reduced features)
# -----------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y  # keep class proportions (recommended)
)

train_data = X_train
test_data = X_test
train_labels = y_train
test_labels = y_test

print("\n--- Data Splitting Results ---")
print(f"Total Samples: {X_reduced.shape[0]}")
print(f"Training Samples: {X_train.shape[0]}")
print(f"Testing Samples: {X_test.shape[0]}")

# -----------------------
# 4) Build smaller quantum circuit
# -----------------------
feature_map = ZZFeatureMap(feature_dimension=N_FEATURES, reps=FEATURE_MAP_REPS)
var_form = RealAmplitudes(num_qubits=N_FEATURES, reps=VAR_ANSATZ_REPS)
circuit = feature_map.compose(var_form)
circuit.measure_all()

# Optional: visualize circuit
try:
    circuit.decompose().draw(output="mpl", fold=-1, style="iqp")
except Exception:
    pass

# -----------------------
# 5) Helpers: instantiate circuit, compute probabilities, robust statevector handling
# -----------------------
def circuit_instance(data_point, variational):
    """Assign parameters for one data point + variational parameters."""
    params = {}
    # feature_map.parameters and var_form.ordered_parameters exist in Qiskit circuits
    # We'll map first len(data_point) parameters to features and remaining to variational
    # NOTE: ordering must match circuit.parameters order; we map by feature_map and var_form lists
    # First assign feature map parameters:
    try:
        for i, p in enumerate(feature_map.parameters):
            params[p] = float(data_point[i])
    except Exception:
        # fallback: assign sequentially
        ps = list(circuit.parameters)
        for i in range(len(data_point)):
            params[ps[i]] = float(data_point[i])
    # Then assign variational parameters
    try:
        for i, p in enumerate(var_form.ordered_parameters):
            params[p] = float(variational[i])
    except Exception:
        # fallback: assign remaining parameters in circuit
        ps = list(circuit.parameters)
        for i, p in enumerate(variational, start=len(data_point)):
            params[ps[i]] = float(p)
    return circuit.assign_parameters(params)

def counts_from_result_entry(res_entry):
    """
    Try to extract measured counts from a sampler result entry.
    If counts are not present, but we can access a statevector, convert amplitudes -> probabilities -> counts-like dict.
    """
    # Try common locations (these depend on Qiskit version); use try/except for robustness
    try:
        # Try data.meas.get_counts() like in your original code
        counts = res_entry.data.meas.get_counts()
        if counts is not None:
            return counts
    except Exception:
        pass

    try:
        # Some runtimes store 'counts' directly
        if hasattr(res_entry, "data") and hasattr(res_entry.data, "counts"):
            counts = res_entry.data.counts
            if counts is not None:
                return counts
    except Exception:
        pass

    # If no counts available, try to extract statevector amplitudes
    try:
        # statevector might be in res_entry.data.statevector or res_entry.data.state
        sv = None
        if hasattr(res_entry.data, "statevector"):
            sv = res_entry.data.statevector
        elif hasattr(res_entry.data, "state"):
            sv = res_entry.data.state
        elif hasattr(res_entry.data, "statevector") is False and isinstance(res_entry.data, dict):
            # sometimes res_entry.data is a dict with 'statevector' key
            sv = res_entry.data.get("statevector", None)

        if sv is None:
            # sometimes the result object has .statevector attribute directly
            sv = getattr(res_entry, "statevector", None)

        if sv is not None:
            # sv expected as complex amplitudes array-like
            amps = np.asarray(sv)
            probs = np.abs(amps) ** 2
            # Build counts-like dict by mapping probabilities to computational basis bitstrings
            nstates = len(probs)
            nqubits = int(np.round(np.log2(nstates)))
            counts = {}
            shots_equiv = 1_000_000  # use large pseudo-shot to convert probs -> counts
            for idx, p in enumerate(probs):
                if p <= 0:
                    continue
                bitstr = format(idx, "0{}b".format(nqubits))[::-1]  # reverse if your convention differs
                counts[bitstr] = int(np.round(p * shots_equiv))
            if counts:
                return counts
    except Exception:
        pass

    # Fallback: return uniform tiny counts to avoid crashes
    return {"0" * N_FEATURES: 1}

def labels_probability_from_counts(counts_dict):
    """
    Convert counts of bitstrings -> probabilities for label 0 / 1 using parity labeling.
    parity = (sum of bits + 1) % 2  (keeps your original mapping)
    returns {0: p0, 1: p1}
    """
    shots = sum(counts_dict.values()) if counts_dict else 1
    probs = {0: 0.0, 1: 0.0}
    for bitstring, c in counts_dict.items():
        # Normalize bitstring: ensure it's a string of bits
        bs = str(bitstring)
        # In case bitstrings have spaces or 'b', strip
        bs = "".join(ch for ch in bs if ch in ("0", "1"))
        bit_sum = sum(int(k) for k in bs)
        label = (bit_sum + 1) % 2
        probs[label] += c / shots
    # Safety: normalize
    s = probs[0] + probs[1]
    if s <= 0:
        return {0: 0.5, 1: 0.5}
    return {0: probs[0] / s, 1: probs[1] / s}

def classification_probability_simulation(data_array, variational):
    """
    For each data point, instantiate its circuit, run the StatevectorSampler, and compute classification probabilities.
    Returns list of dicts: [{0: p0, 1: p1}, ...]
    """
    circuits = [circuit_instance(d, variational) for d in data_array]
    sampler = StatevectorSampler()
    results = sampler.run(circuits).result()

    classifications = []
    # The result object may be iterable or indexable; handle both patterns
    for i in range(len(circuits)):
        try:
            res_entry = results[i]
        except Exception:
            # some versions keep results._result_list or results.data
            try:
                res_entry = results._result_list[i]
            except Exception:
                res_entry = results
        counts = counts_from_result_entry(res_entry)
        probs = labels_probability_from_counts(counts)
        classifications.append(probs)
    return classifications

# -----------------------
# 6) Loss, cost, and objective
# -----------------------
def cross_entropy_loss(classification_probs, expected_label):
    # classification_probs is a dict {0: p0, 1: p1}
    p = classification_probs.get(expected_label, 0.0)
    return -np.log(p + 1e-12)

def cost_function(data, labels, variational):
    classifications = classification_probability_simulation(data, variational)
    total = 0.0
    for i, probs in enumerate(classifications):
        total += cross_entropy_loss(probs, labels[i])
    return total / len(data)

def objective_function(variational):
    return cost_function(train_data, train_labels, variational)

# -----------------------
# 7) Optimizer and training
# -----------------------
class OptimizerLog:
    def __init__(self):
        self.evaluations = []
        self.parameters = []
        self.costs = []
    def update(self, evaluation, parameter, cost, _stepsize, _accept):
        self.evaluations.append(evaluation)
        self.parameters.append(parameter)
        self.costs.append(cost)

log = OptimizerLog()
optimizer = SPSA(maxiter=SPSA_MAXITER, callback=log.update)

# smaller random initial point (helps break symmetry)
initial_point = 0.01 * np.random.randn(var_form.num_parameters)

print("\nStarting optimization (this may take a while)...")
result = optimizer.minimize(objective_function, initial_point)
opt_var = result.x
opt_value = result.fun
print(f"Optimization finished. Best cost: {opt_value:.6f}")

# Plot optimization progress
plt.figure(figsize=(8,4))
plt.plot(log.evaluations, log.costs, marker='o', markersize=3)
plt.xlabel("SPSA Step")
plt.ylabel("Cost")
plt.grid(True)
plt.title("Optimizer progress")
plt.show()

# -----------------------
# 8) Evaluation utilities: confusion matrix & test
# -----------------------
def create_conf_mat(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    # normalize per-row
    cmn = cm.astype(float) / (cm.sum(axis=1)[:, np.newaxis] + 1e-12) * 100

    # custom colormap (white -> purple-like)
    colors = [(1, 1, 1), (98/255, 35/255, 223/255)]
    cmap = LinearSegmentedColormap.from_list("custom", colors, N=100)

    fig, ax = plt.subplots()
    cax = ax.matshow(cmn, cmap=cmap)
    fig.colorbar(cax)
    for i in range(cmn.shape[0]):
        for j in range(cmn.shape[1]):
            color = 'white' if i == j else 'black'
            ax.text(j, i, f"{cmn[i, j]:.1f}%", va='center', ha='center', color=color)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    plt.show()

def test_classifier(data, labels, variational):
    probs_list = classification_probability_simulation(data, variational)
    preds = [0 if p[0] >= p[1] else 1 for p in probs_list]
    accuracy = np.mean(np.array(preds) == np.array(labels))
    print(f"Simulated Test Accuracy: {accuracy*100:.2f}%")
    create_conf_mat(labels, preds, "Simulated Evaluation")
    return accuracy, preds

# Evaluate on test set
accuracy_sim, predictions_sim = test_classifier(test_data, test_labels, opt_var)

# Optionally: evaluate on training set (to detect overfitting)
accuracy_train, preds_train = test_classifier(train_data, train_labels, opt_var)

print("Done.")


SUCCESS: Data loaded from Heart_data.csv

--- Data Head ---
   Age  Gender  BloodPressure  Cholesterol  HeartRate  QuantumPatternFeature  \
0   68       1            105          191        107               8.362241   
1   58       0             97          249         89               9.249002   
2   44       0             93          190         82               7.942542   
3   72       1             93          183        101               6.495155   
4   37       0            145          166        103               7.653900   

   HeartDisease  
0             1  
1             0  
2             1  
3             1  
4             1  

--- Array Dimensions ---
Features (X) shape: (500, 6)
Labels (y) shape: (500,)

After PCA: X_reduced shape = (500, 3)
Using N_FEATURES (qubits) = 3

--- Data Splitting Results ---
Total Samples: 500
Training Samples: 400
Testing Samples: 100

Starting optimization (this may take a while)...
